## Example 1: read the version from package.json

In [ ]:
import importlib.metadata
import hatch_nodejs_version
from hatch_nodejs_version import hooks
import json
import os
import tempfile
from hatch_nodejs_version.version_source import NodeJSVersionSource


def write_package_json(directory, data, name="package.json"):
    path = os.path.join(directory, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    return path


# hatchling finds the version source through the plugin's hook.
assert hooks.hatch_register_version_source() is NodeJSVersionSource
assert NodeJSVersionSource.PLUGIN_NAME == "nodejs"
assert hatch_nodejs_version.__version__ == importlib.metadata.version("hatch-nodejs-version")

with tempfile.TemporaryDirectory() as root:
    write_package_json(root, {"name": "test-package", "version": "1.2.3"})
    data = NodeJSVersionSource(root, {}).get_version_data()
    print(f"version data: {data}")
    assert data == {"version": "1.2.3"}

## Example 2: a NodeJS pre-release becomes a PEP 440 pre-release

In [ ]:
import json
import os
import tempfile
from hatch_nodejs_version.version_source import NodeJSVersionSource
from packaging.version import Version


def write_package_json(directory, data, name="package.json"):
    path = os.path.join(directory, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    return path


with tempfile.TemporaryDirectory() as root:
    write_package_json(root, {"name": "my-app", "version": "2.1.0-beta.1", "main": "index.js"})
    version = NodeJSVersionSource(root, {}).get_version_data()["version"]
    print(f"package.json 2.1.0-beta.1 -> Python version {version} ({Version(version)})")
    assert Version(version) == Version("2.1.0b1")
    assert Version(version).is_prerelease

## Example 3: the `path` option reads another package.json

In [ ]:
import json
import os
import tempfile
from hatch_nodejs_version.version_source import NodeJSVersionSource


def write_package_json(directory, data, name="package.json"):
    path = os.path.join(directory, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    return path


with tempfile.TemporaryDirectory() as root:
    write_package_json(root, {"name": "root-package", "version": "0.1.0"})
    write_package_json(root, {"name": "@scope/web", "version": "0.5.2", "private": True}, name="web/package.json")

    default = NodeJSVersionSource(root, {})
    nested = NodeJSVersionSource(root, {"path": "web/package.json"})
    print(f"default path {default.path!r}: {default.get_version_data()['version']}")
    print(f"configured path {nested.path!r}: {nested.get_version_data()['version']}")
    assert default.path == "package.json" and default.get_version_data() == {"version": "0.1.0"}
    assert nested.get_version_data() == {"version": "0.5.2"}

    try:
        NodeJSVersionSource(root, {"path": 5}).path
    except TypeError as error:
        print(f"rejected a non-string path: {error}")
    else:
        raise AssertionError("a non-string path was accepted")

## Example 4: converting version strings in both directions

In [ ]:
from hatch_nodejs_version.version_source import NodeJSVersionSource
from packaging.version import Version

node_to_python = {
    "1.0.0": "1.0.0",
    "2.1.3-alpha.1": "2.1.3a1",
    "3.0.0-rc.2+build.123": "3.0.0rc2+build.123",
    "10.15.0": "10.15.0",
    "1.0.0-dev.3": "1.0.0.dev3",
}
for node_version, expected in node_to_python.items():
    python_version = NodeJSVersionSource.node_version_to_python(node_version)
    print(f"{node_version:22} -> {python_version}")
    assert Version(python_version) == Version(expected), (node_version, python_version)

# And back: a Python version written into package.json
assert NodeJSVersionSource.python_version_to_node("1.1.0rc1") == "1.1.0-rc1"
assert NodeJSVersionSource.python_version_to_node("2.0.0") == "2.0.0"

# A version without a patch number is not a NodeJS version.
try:
    NodeJSVersionSource.node_version_to_python("1.0")
except ValueError as error:
    print(f"rejected: {error}")
else:
    raise AssertionError("'1.0' was accepted")

## Example 5: a missing package.json raises OSError, and set_version writes the file

In [ ]:
import json
import os
import tempfile
from hatch_nodejs_version.version_source import NodeJSVersionSource


def write_package_json(directory, data, name="package.json"):
    path = os.path.join(directory, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    return path


with tempfile.TemporaryDirectory() as root:
    try:
        NodeJSVersionSource(root, {}).get_version_data()
    except OSError as error:
        print(f"missing package.json: {error}")
    else:
        raise AssertionError("a version was read without a package.json")

    path = write_package_json(root, {"name": "test", "version": "1.0.0"})
    source = NodeJSVersionSource(root, {})
    assert source.get_version_data() == {"version": "1.0.0"}

    # set_version stores the NodeJS form of the version and keeps the other fields.
    source.set_version("1.1.0rc1", {})
    with open(path) as f:
        stored = json.load(f)
    print(f"package.json after set_version: {stored}")
    assert stored == {"name": "test", "version": "1.1.0-rc1"}
    assert NodeJSVersionSource(root, {}).get_version_data() == {"version": "1.1.0rc1"}